In [ ]:
import kagglehub
import os
import pandas as pd


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

golden = os.path.join(path,'Q3_data.csv')

df = pd.read_csv(golden)

print("the is shape is :",df.shape)

In [ ]:
# Task 2: Write your code here:

print('the dataset first few rows \n',df.head())

In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)



df.drop(columns=['D_87','D_88','B_39','D_110','D_111','D_108','B_42','D_73','D_135','D_136'])

In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
categorical_cols=df.select_dtypes(include=["object"]).columns

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

print("Categorical Columns:", list(categorical_cols))


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import MinMaxScaler

features = df.columns  # DON'T SCALE THE TARGET

scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])
df.head()


In [ ]:
# Task 5: Write your code here:
def softmax(z):
  z_shifted = z - np.max(z, axis=1, keepdims=True)
  exp_z = np.exp(z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [ ]:
X = df.drop("P_2", axis=1).astype(float)
y = df['P_2'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

# Task 1: Write your code here:
from catboost import CatBoostRegressor
model=( CatBoostRegressor(verbose=0))


kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]



    # Train
model.fit(X_train, y_train)

    # Predict
y_pred = model.predict(X_test)

    # Calculate metrics
mse = sklearn_mse(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

    # Store results
all_results[model_name]["mse"].append(mse)
all_results[model_name]["rmse"].append(rmse)
all_results[model_name]["r2"].append(r2)

In [ ]:
# Task 1: Write your code here:
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mse = sklearn_mse(y, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y, baseline_pred)

print(f"Baseline MSE (using mean target): {baseline_mse:.4f}")
print(f"Baseline RMSE (using mean target): {baseline_rmse:.4f}")
print(f"Baseline R2 (using mean target): {baseline_r2:.4f}")


In [ ]:
# Task 2: Write your code here:
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: